# Eksperimen Spark MLlib PaySim — versi sumber diperbaiki (11 kolom)

Notebook ini menjalankan eksperimen **Random Forest menggunakan Spark MLlib** dari paket PaySim yang sudah diperbaiki. Skrip sumber sekarang membaca seluruh 11 kolom CSV pada posisi yang benar.

**Yang diuji:** pemeriksaan data, verifikasi aturan dan unit test, eksperimen model, lalu penyimpanan hasil agar mudah diperiksa.

**Batas interpretasi:** PaySim adalah dataset sintetis mobile money, bukan data transaksi EDC/Base24 bank. Hasil tinggi bukan bukti performa yang sama pada transaksi bank nyata. Spark di Colab berjalan lokal (`local[2]`/mode lokal runtime), bukan bukti skalabilitas cluster multi-node.

Jalankan sel **berurutan dari atas ke bawah**. Jangan lompat ke sel eksperimen sebelum instalasi, dataset, dan tes awal berhasil.


## 1. Unggah dan ekstrak paket source code ASG2

Unggah **`ASG2_Source_Code_PaySim.zip`**, yaitu paket source code yang menyertakan skrip eksperimen, unit tests, notebook, serta output eksperimen yang sudah tersimpan.

**Dataset CSV mentah tidak disertakan di dalam ZIP** agar ukuran paket tetap kecil. Unduh dataset PaySim dari Kaggle (tautan ada di README proyek), lalu simpan CSV bernama `PS_20174392719_1491204439457_log.csv` di Google Drive. Sel berikutnya akan membantu menyalinnya ke folder proyek.

Notebook ini merupakan notebook sumber yang dapat dijalankan ulang. Output eksperimen yang sudah pernah dijalankan disimpan sebagai file terpisah di folder `outputs/`, termasuk metrik, prediksi, log, dan model Spark MLlib. Sel notebook tidak menyimpan output run lama agar hasil baru tidak tercampur dengan hasil sebelumnya.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Unggah tepat satu ZIP paket proyek v2.')

zip_path = Path('/content') / zip_names[0]
extract_root = Path('/content/paysim_experiment_v2')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise RuntimeError(f'ZIP rusak pada file: {bad_member}')
    zf.extractall(extract_root)

matches = list(extract_root.rglob('run_paysim_spark.py'))
if len(matches) != 1:
    raise FileNotFoundError(f'Diharapkan satu run_paysim_spark.py, ditemukan {len(matches)}. Pastikan ZIP v2 yang benar diunggah.')
PROJECT_DIR = matches[0].parent
DATA_PATH = PROJECT_DIR / 'data' / 'PS_20174392719_1491204439457_log.csv'
print('Folder proyek:', PROJECT_DIR)
print('Skrip Spark:', (PROJECT_DIR / 'run_paysim_spark.py').exists())
print('Dataset sudah ada di paket:', DATA_PATH.exists())


## 2. Pastikan dataset tersedia

Paket source code yang digunakan tidak menyertakan CSV mentah. Simpan file `PS_20174392719_1491204439457_log.csv` yang diunduh dari Kaggle ke Google Drive. Notebook akan mencari nama file tersebut secara otomatis. Jika ditemukan lebih dari satu salinan, isi `DATA_CSV_PATH` pada sel berikut dengan path yang benar.

In [ ]:
from pathlib import Path
import shutil

# Opsional: isi dengan path lengkap CSV di Google Drive jika pencarian otomatis menemukan lebih dari satu.
# Contoh: '/content/drive/MyDrive/BigData/PS_20174392719_1491204439457_log.csv'
DATA_CSV_PATH = ''

if not DATA_PATH.exists():
    from google.colab import drive
    drive.mount('/content/drive')

    if DATA_CSV_PATH.strip():
        source_csv = Path(DATA_CSV_PATH.strip())
        if not source_csv.is_file():
            raise FileNotFoundError(f'CSV tidak ditemukan di DATA_CSV_PATH: {source_csv}')
    else:
        drive_root = Path('/content/drive/MyDrive')
        expected_name = 'PS_20174392719_1491204439457_log.csv'
        candidates = list(drive_root.rglob(expected_name)) if drive_root.exists() else []
        if len(candidates) == 0:
            raise FileNotFoundError(
                'CSV PaySim belum ditemukan di MyDrive. Simpan file bernama '
                'PS_20174392719_1491204439457_log.csv ke Google Drive, lalu jalankan ulang sel ini. '
                'Alternatif: isi DATA_CSV_PATH dengan lokasi CSV yang tepat.'
            )
        if len(candidates) > 1:
            print('Ditemukan beberapa CSV:')
            for candidate in candidates:
                print(' -', candidate)
            raise ValueError('Isi DATA_CSV_PATH dengan salah satu path di atas, lalu jalankan ulang sel ini.')
        source_csv = candidates[0]

    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_csv, DATA_PATH)

print('Dataset siap:', DATA_PATH)
print('Ukuran dataset (MB):', round(DATA_PATH.stat().st_size / (1024 * 1024), 1))


## 3. Periksa Java

Spark 4.x memerlukan Java 17 atau lebih baru. Sel ini memeriksa versi yang tersedia dan mencoba memasang Java 17 jika belum ada atau terlalu lama.


In [ ]:
import subprocess, shutil, re, os
from pathlib import Path

def get_java_info():
    java_bin = shutil.which('java')
    if not java_bin:
        return None, None, None
    result = subprocess.run([java_bin, '-version'], capture_output=True, text=True)
    output = result.stderr + result.stdout
    match = re.search(r'version \"(\d+)', output)
    major = int(match.group(1)) if match else None
    resolved = Path(java_bin).resolve()
    java_home = resolved.parent.parent
    return major, output.strip(), str(java_home)

major, java_info, java_home = get_java_info()
if major is None or major < 17:
    print('Java 17+ belum tersedia; mencoba memasang OpenJDK 17...')
    subprocess.run('apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless', shell=True, check=True)
    subprocess.run(['update-alternatives', '--set', 'java', '/usr/lib/jvm/java-17-openjdk-amd64/bin/java'], check=False)
    major, java_info, java_home = get_java_info()

print(java_info or 'Versi Java tidak dapat dibaca')
if major is None or major < 17:
    raise RuntimeError('Java 17 atau lebih baru dibutuhkan. Jika gagal, kirim keluaran sel ini.')
os.environ['JAVA_HOME'] = java_home
os.environ['PATH'] = f"{java_home}/bin:" + os.environ.get('PATH', '')
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
print('Pemeriksaan Java lulus. JAVA_HOME =', os.environ['JAVA_HOME'])


## 4. Pasang pustaka eksperimen

Tunggu sampai instalasi selesai sebelum melanjutkan. Notebook menggunakan PySpark 4.0.x, pandas, NumPy, dan scikit-learn untuk menghitung metrik evaluasi.


In [ ]:
import sys, subprocess

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pyspark>=4.0,<4.1', 'pandas>=2.1,<3.0', 'numpy>=1.26', 'scikit-learn'
], check=True)

import pyspark, pandas as pd, numpy as np, sklearn
print('Python:', sys.version.split()[0])
print('PySpark:', pyspark.__version__)
print('pandas:', pd.__version__)
print('NumPy:', np.__version__)
print('scikit-learn:', sklearn.__version__)


## 5. Tes Spark kecil

Ini hanya tes bahwa Java dan Spark dapat berkomunikasi. Output yang diharapkan adalah `Jumlah baris tes: 5`. Ini **belum** menjalankan eksperimen fraud.


In [ ]:
from pyspark.sql import SparkSession

spark_test = (SparkSession.builder
              .appName('PaySimSparkSmokeTestV2')
              .master('local[2]')
              .config('spark.sql.shuffle.partitions', '4')
              .getOrCreate())
try:
    print('Spark version:', spark_test.version)
    n = spark_test.range(5).count()
    print('Jumlah baris tes:', n)
    assert n == 5, 'Tes Spark tidak menghasilkan 5 baris.'
finally:
    spark_test.stop()
print('Tes Spark lulus.')


## 6. Validasi header CSV dan contoh data

Sebelum membaca jutaan baris, kita periksa header dan beberapa baris pertama. CSV yang benar harus memiliki **11 kolom** dalam urutan PaySim standar.


In [ ]:
import pandas as pd

EXPECTED_COLUMNS = [
    'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
    'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud'
]
preview = pd.read_csv(DATA_PATH, nrows=5)
print('Jumlah kolom:', len(preview.columns))
print('Nama kolom:', preview.columns.tolist())
if preview.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError('Header CSV tidak sesuai dengan 11 kolom PaySim yang diharapkan. Periksa file dataset.')
print('Lima baris pertama:')
display(preview)
print('Header CSV valid.')


## 7. Verifikasi aturan dan unit test

Aturan bisnis diuji terpisah memakai skenario sintetis. Bagian ini tidak melatih model machine learning. Targetnya adalah checksum aturan tetap sama dan semua 15 unit test lulus.


In [ ]:
import subprocess, sys

print('=== Verifikasi konfigurasi aturan beku ===')
subprocess.run([sys.executable, 'verify_rules_freeze.py'], cwd=PROJECT_DIR, check=True)

print('=== Unit tests ===')
test_result = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=PROJECT_DIR
)
if test_result.returncode != 0:
    raise RuntimeError('Ada unit test gagal. Periksa error sebelum melanjutkan ke eksperimen model.')
print('Verifikasi aturan dan unit test selesai.')


## 8. Jalankan eksperimen Random Forest dengan Spark MLlib

Skrip menggunakan pembagian kronologis: *train* sampai `step=520`, *validation* pada `step=521–600`, dan *test* pada `step>600`. Threshold dipilih dari *validation*, kemudian dievaluasi pada *test*. Jangan memilih ulang threshold berdasarkan hasil *test*.

Eksperimen dapat memakan waktu dan penggunaan memori cukup tinggi. Tunggu sampai pesan `SPARK TEST METRICS` muncul. Jika proses gagal, notebook menyimpan keluaran ke `outputs/spark/colab_run.log`.


In [ ]:
import subprocess, sys, shutil
from pathlib import Path

spark_out = PROJECT_DIR / 'outputs' / 'spark'
# Bersihkan hanya keluaran Spark lama supaya hasil yang ditampilkan pasti berasal dari run ini.
if spark_out.exists():
    shutil.rmtree(spark_out)
spark_out.mkdir(parents=True, exist_ok=True)
log_path = spark_out / 'colab_run.log'

with log_path.open('w', encoding='utf-8') as log_file:
    proc = subprocess.Popen(
        [sys.executable, '-u', 'run_paysim_spark.py'],
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        print(line, end='')
        log_file.write(line)
    return_code = proc.wait()

RUN_SUCCESS = (return_code == 0) and (spark_out / 'final_test_metrics.json').exists()
print('Status eksperimen:', 'BERHASIL' if RUN_SUCCESS else 'GAGAL')
print('Kode keluar:', return_code)
print('Log:', log_path)
if not RUN_SUCCESS:
    print('Jangan menyimpulkan metrik jika eksperimen gagal. Unduh ZIP hasil pada sel terakhir untuk memeriksa log.')


## 9. Tampilkan hasil

Jika run berhasil, sel ini menampilkan metrik *test*, confusion matrix, pembagian data kronologis, dan trade-off jumlah alert. Perhatikan jumlah TP/FP/FN/TN serta *Average Precision*, jangan hanya melihat akurasi.


In [ ]:
import json, pandas as pd
from IPython.display import display

metrics_path = PROJECT_DIR / 'outputs' / 'spark' / 'final_test_metrics.json'
if not metrics_path.exists():
    print('File metrik belum terbentuk. Buka outputs/spark/colab_run.log untuk melihat error.')
else:
    result = json.loads(metrics_path.read_text(encoding='utf-8'))
    test = result.get('test_metrics_at_validation_threshold', {})
    fields = [
        'rows', 'fraud_actual', 'nonfraud_actual', 'threshold', 'alerts',
        'true_positive', 'false_positive', 'true_negative', 'false_negative',
        'precision', 'recall', 'f1', 'accuracy', 'roc_auc', 'average_precision'
    ]
    display(pd.DataFrame([{'metrik': k, 'nilai': test.get(k)} for k in fields]))
    print('Ringkasan split waktu:')
    if 'split' in result:
        display(pd.DataFrame(result['split']))
    cm_path = PROJECT_DIR / 'outputs' / 'spark' / 'confusion_matrix.csv'
    if cm_path.exists():
        print('Confusion matrix:')
        display(pd.read_csv(cm_path, index_col=0))
    budget_path = PROJECT_DIR / 'outputs' / 'spark' / 'test_alert_budget_metrics.csv'
    if budget_path.exists():
        print('Trade-off jumlah alert pada data test:')
        display(pd.read_csv(budget_path))
    print('Catatan:', result.get('warning', ''))


## 10. Kemas dan unduh hasil eksperimen

ZIP ini menyertakan hasil, log, dan skrip yang dijalankan agar run bisa diperiksa. Jika eksperimen gagal, tetap unduh ZIP lalu bagikan `colab_run.log` beserta pesan error. Jika berhasil, simpan ZIP sebagai bukti hasil eksperimen.


In [ ]:
from pathlib import Path
import zipfile, subprocess, sys
from google.colab import files

result_zip = Path('/content/spark_mllib_paysim_v2_results.zip')
env_file = PROJECT_DIR / 'outputs' / 'spark' / 'experiment_environment.txt'
java_result = subprocess.run(['java', '-version'], capture_output=True, text=True)
env_file.write_text(
    'Python: ' + sys.version.split()[0] + '\n' +
    'PySpark: ' + __import__('pyspark').__version__ + '\n' +
    'Java:\n' + (java_result.stderr or java_result.stdout) + '\n' +
    'Run success: ' + str(globals().get('RUN_SUCCESS', False)) + '\n',
    encoding='utf-8'
)

with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    if spark_out.exists():
        for p in spark_out.rglob('*'):
            if p.is_file():
                zf.write(p, arcname=str(Path('outputs/spark') / p.relative_to(spark_out)))
    zf.write(PROJECT_DIR / 'run_paysim_spark.py', arcname='source/run_paysim_spark.py')
    zf.write(PROJECT_DIR / 'README.md', arcname='source/README.md')

print('ZIP hasil:', result_zip)
print('Ukuran (MB):', round(result_zip.stat().st_size / (1024 * 1024), 2))
files.download(str(result_zip))


## Jika terjadi error

1. Jangan ubah model atau threshold dulu.
2. Jalankan sel terakhir agar ZIP hasil/log terunduh (jika runtime masih aktif).
3. Kirim pesan error lengkap dan file `colab_run.log` untuk diperiksa.

Untuk laporan, hasil ini harus dijelaskan sebagai performa pada **dataset PaySim sintetis**. Selain itu, `amount == oldbalanceOrg` merupakan pola yang sangat khas pada simulator tersebut, sehingga metrik tinggi bukan bukti bahwa metode akan mendapatkan hasil yang sama pada transaksi bank nyata.
